# LAB | Whisper STT Implementation Lab

Author: Louise Plessis

## Step 1: Setting Up the Environment

In [27]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file in current directory
api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

Key loaded: True


In [28]:
import subprocess
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(result.stdout[:100])

ffmpeg version 8.1.2 Copyright (c) 2000-2026 the FFmpeg developers
built with Apple clang version 21


## Step 2: Downloading Sample Meeting Audio

In [29]:
import os

filepath = "audio/california_moon_landing.mp3"

print("File exists:", os.path.exists(filepath))
print("File size (KB):", os.path.getsize(filepath) / 1024)

with open(filepath, "rb") as f:
    header = f.read(20)
print("First 20 bytes:", header)

File exists: True
File size (KB): 909.6240234375
First 20 bytes: b'\xff\xfb\x90\xc4\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


File exists: True
File size (KB): 909.6240234375
First 20 bytes: b'\xff\xfb\x90\xc4\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [30]:
import librosa

y, sr = librosa.load(filepath, sr=None)
duration = librosa.get_duration(y=y, sr=sr)

print(f"Sample rate: {sr} Hz")
print(f"Duration: {duration:.2f} seconds")
print(f"Number of samples: {len(y)}")

Sample rate: 44100 Hz
Duration: 86.20 seconds
Number of samples: 3801347


Sample rate: 44100 Hz
Duration: 86.20 seconds
Number of samples: 3801347

## Step 3: Basic Transcription (Without Chunking)

In [31]:
# Basic transcription using Whisper
from openai import OpenAI
import os

# Initialize the client (reads OPENAI_API_KEY from your environment automatically)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

filepath = "audio/california_moon_landing.mp3"

with open(filepath, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print("Transcribed text:")
print(transcription.text)

Transcribed text:
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and all. And

Transcribed text:

It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and all. And they did probe for some samples. And just what was going to come of that, I don't know. I'll be glad to read it.

In [32]:
# Json response with segments and timestamps
with open(filepath, "rb") as audio_file:
    transcription_verbose = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="verbose_json"
    )

print("Detected language:", transcription_verbose.language)
print("Duration:", transcription_verbose.duration)
print("\nFirst segment example:")
print(transcription_verbose.segments[0])

Detected language: english
Duration: 86.19000244140625

First segment example:
TranscriptionSegment(id=0, avg_logprob=-0.3732287585735321, compression_ratio=1.5822222232818604, end=3.559999942779541, no_speech_prob=0.0020829029381275177, seek=0, start=0.0, temperature=0.0, text=' It was rather interesting just to watch them gathering', tokens=[50364, 467, 390, 2831, 1880, 445, 281, 6858, 339, 552, 13519, 50542])


## Step 4: Transcription with Prompts 

In [42]:
filepath = "audio/california_moon_landing.mp3"

prompt_context = (
    "This is a 1969 conversation about the Apollo moon landing. "
    "Topics include astronauts, rendezvous procedures, lunar surface exploration, "
    "and NASA space missions."
)

with open(filepath, "rb") as audio_file:
    transcription_prompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=prompt_context
    )

print("Prompted transcription:")
print(transcription_prompted.text)

Prompted transcription:
It was rather interesting just to watch them gathering their materials and bouncing around, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Yes, I'll be so glad when they land back now, but I think that's pretty well a fact because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? They've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused, so that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and all, and they did pro

Prompted transcription:
It was rather interesting just to watch them gathering their materials and bouncing around. What do they call it? Kangaroo walk? Something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Oh yes, I'll be so glad when they land back now. I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused. That's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area when you think of it. They just gathered rocks and samples of soil and all, and they did probe for some samples. Just what's going to come of that, I don't know. I'd be glad to read it.

## Step 5: Transcription Without Prompts

In [35]:
## Step 5: Transcription Without Prompts 
filepath = "audio/california_moon_landing.mp3"

with open(filepath, "rb") as audio_file:
    transcription_unprompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
        # note: no prompt parameter passed
    )

print("Unprompted transcription:")
print(transcription_unprompted.text)

Unprompted transcription:
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and 

#### Prompted vs. Unprompted Transcription Comparison

**Prompt used:** "This is a 1969 conversation about the Apollo moon landing. Topics include
astronauts, rendezvous procedures, lunar surface exploration, and NASA space missions." "Speaker is from Weaverville, CA; she is a 72-year-old white woman with a high-school education. 

##### Key differences

| Location in audio | Unprompted | Prompted | Notes |
|---|---|---|---|
| "get back to ___" | **Maine** | **Earth** | Prompt corrected a clear mishearing — "Maine" makes no sense in context, "Earth" does. Strongest example of prompt improving accuracy. |
| Future plans question | "Do you imagine them **sending up ships**?" | "Do you imagine them **standing up**?" | Here the *unprompted* version is more coherent. The prompted version arguably got worse — "standing up" doesn't fit the context as well as "sending up ships" does for a question about future NASA missions. |
| Closing line | "**I'll** be glad to read it" | "**I'd** be glad to read it" | Minor, likely inconsequential (ambiguous audio, either could be correct). |
| Punctuation/pacing | Slightly more run-on, extra "So"/"And" at sentence starts | Slightly cleaner sentence breaks | Prompted version is easier to read but not more *accurate* per se. |

##### Takeaway
The prompt context helped in one clear case (Maine → Earth) but didn't uniformly improve results — it introduced a plausible-but-wrong word ("standing up") in a spot where the unprompted model got the more sensible answer ("sending up ships"). This suggests prompts bias the model toward domain vocabulary but can occasionally override a correct guess with a contextually "expected" one that doesn't actually fit. Prompting is a net positive but not a guarantee of accuracy on every segment.

## Step 6: Implementing Audio Chunking


In [36]:
from pydub import AudioSegment
import os

filepath = "audio/california_moon_landing.mp3"
chunk_length_minutes = 0.5

audio = AudioSegment.from_mp3(filepath)
duration_ms = len(audio)
chunk_length_ms = int(chunk_length_minutes * 60 * 1000)

os.makedirs("audio/chunks", exist_ok=True)

chunks = []
for i, start_ms in enumerate(range(0, duration_ms, chunk_length_ms)):
    end_ms = min(start_ms + chunk_length_ms, duration_ms)
    chunk = audio[start_ms:end_ms]
    chunk_path = f"audio/chunks/chunk_{i}.mp3"
    chunk.export(chunk_path, format="mp3")
    chunks.append({"path": chunk_path, "start_ms": start_ms, "end_ms": end_ms})
    print(f"Created {chunk_path}: {start_ms/1000:.1f}s - {end_ms/1000:.1f}s")

print(f"\nTotal chunks created: {len(chunks)}")

Created audio/chunks/chunk_0.mp3: 0.0s - 30.0s
Created audio/chunks/chunk_1.mp3: 30.0s - 60.0s
Created audio/chunks/chunk_2.mp3: 60.0s - 86.2s

Total chunks created: 3


Created audio/chunks/chunk_0.mp3: 0.0s - 30.0s
Created audio/chunks/chunk_1.mp3: 30.0s - 60.0s
Created audio/chunks/chunk_2.mp3: 60.0s - 86.2s

Total chunks created: 3

## Step 7: Transcribing Chunks with Timestamps


In [37]:
import time

all_segments = []
full_text_parts = []

for chunk_info in chunks:
    chunk_path = chunk_info["path"]
    offset_seconds = chunk_info["start_ms"] / 1000

    with open(chunk_path, "rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json"
        )

    full_text_parts.append(result.text)

    for seg in result.segments:
        all_segments.append({
            "start": seg.start + offset_seconds,
            "end": seg.end + offset_seconds,
            "text": seg.text
        })

    print(f"Transcribed {chunk_path} ({len(result.segments)} segments)")
    time.sleep(1)

full_transcript = " ".join(full_text_parts)

print(f"\nTotal segments across all chunks: {len(all_segments)}")
print(f"\nFull combined transcript:\n{full_transcript}")

Transcribed audio/chunks/chunk_0.mp3 (7 segments)
Transcribed audio/chunks/chunk_1.mp3 (7 segments)
Transcribed audio/chunks/chunk_2.mp3 (5 segments)

Total segments across all chunks: 19

Full combined transcript:
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now. I think that's pretty well a fact because they've landed so many safely now. and I feel relief. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, so I... But they say they have rendezvoused, so... That's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending a... I think

Transcribed audio/chunks/chunk_0.mp3 (7 segments)
Transcribed audio/chunks/chunk_1.mp3 (7 segments)
Transcribed audio/chunks/chunk_2.mp3 (5 segments)

Total segments across all chunks: 19

Full combined transcript:
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now, but I think that's pretty well fact because they've landed so many safely now. and I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, so I... But they say they have rendezvoused, so... That's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending a... I think they will. I think they will. And we'll do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and, oh, samples of soil and all. And they did a probe for some samples. And just what was going to come of that, I don't know. I'd be glad to read it.

Note: Fixed-length chunking without overlap can truncate content at chunk boundaries when a sentence spans a cut point (observed: "sending up ships" was cut to "sending a..." at the 60s boundary). Overlap-based padding can mitigate this but introduces its own complexity around deduplicating repeated content — a trade-off between engineering effort and transcription completeness.

## Step 8: Exporting with Timestamps


In [41]:
import json

os.makedirs("output", exist_ok=True)

def format_srt_timestamp(seconds):
    """Convert seconds to SRT format: HH:MM:SS,mmm"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

def format_readable_timestamp(seconds):
    """Convert seconds to readable [MM:SS] format"""
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"[{minutes:02d}:{secs:02d}]"

# --- 1. Human-readable text export ---
txt_path = "output/transcription.txt"
with open(txt_path, "w") as f:
    f.write("INTERVIEW MOON LANDING TRANSCRIPTION\n")
    f.write("=" * 50 + "\n\n")
    for seg in all_segments:
        timestamp = format_readable_timestamp(seg["start"])
        f.write(f"{timestamp} {seg['text'].strip()}\n")
print(f"Saved: {txt_path}")

# --- 2. SRT subtitle export ---
srt_path = "output/transcription.srt"
with open(srt_path, "w") as f:
    for i, seg in enumerate(all_segments, start=1):
        start_ts = format_srt_timestamp(seg["start"])
        end_ts = format_srt_timestamp(seg["end"])
        f.write(f"{i}\n")
        f.write(f"{start_ts} --> {end_ts}\n")
        f.write(f"{seg['text'].strip()}\n\n")
print(f"Saved: {srt_path}")

# --- 3. JSON export (structured, for programmatic use) ---
json_path = "output/transcription.json"
json_data = {
    "source_file": filepath,
    "full_text": full_transcript,
    "segments": [
        {
            "start": round(seg["start"], 2),
            "end": round(seg["end"], 2),
            "text": seg["text"].strip()
        }
        for seg in all_segments
    ]
}
with open(json_path, "w") as f:
    json.dump(json_data, f, indent=2)
print(f"Saved: {json_path}")

print("\nAll exports complete. Preview of each:\n")

print("--- TXT preview ---")
with open(txt_path) as f:
    print("".join(f.readlines()[:6]))

print("--- SRT preview ---")
with open(srt_path) as f:
    print("".join(f.readlines()[:8]))

print("--- JSON preview ---")
print(json.dumps(json_data["segments"][:2], indent=2))

Saved: output/transcription.txt
Saved: output/transcription.srt
Saved: output/transcription.json

All exports complete. Preview of each:

--- TXT preview ---
INTERVIEW MOON LANDING TRANSCRIPTION

[00:00] It was rather interesting just to watch them gathering their materials and bouncing around
[00:07] their, what do they call it, kangaroo walk or something like that.
[00:12] Who named it that?

--- SRT preview ---
1
00:00:00,000 --> 00:00:07,000
It was rather interesting just to watch them gathering their materials and bouncing around

2
00:00:07,000 --> 00:00:12,500
their, what do they call it, kangaroo walk or something like that.


--- JSON preview ---
[
  {
    "start": 0.0,
    "end": 7.0,
    "text": "It was rather interesting just to watch them gathering their materials and bouncing around"
  },
  {
    "start": 7.0,
    "end": 12.5,
    "text": "their, what do they call it, kangaroo walk or something like that."
  }
]


### Whisper STT Implementation — Report
**Author:** Louise Plessis

#### Prompted vs. Unprompted
Both transcribed accurately overall but diverged on ambiguous words: unprompted
misheard a destination as "Maine," prompted corrected it to "Earth." But prompting
isn't strictly better — elsewhere unprompted correctly caught "sending up ships"
while prompted produced the less coherent "standing up." Re-running unprompted
(Step 5) gave identical output to Step 3, confirming the model is deterministic
here — so the differences are a real prompt effect, not noise.

#### Why Chunking
Whisper enforces a 25MB file limit, so long recordings must be split to stay under
it. Chunking also allows parallel processing and isolates failures to one segment
instead of the whole file.

#### Challenges
- **Timestamp continuity**: each chunk's timestamps reset near 0, so each chunk's
  offset must be added back to reconstruct one continuous timeline.
- **Boundary truncation**: a sentence spanning a chunk cut can be lost ("sending up
  ships" became "sending a..."). I tried fixing this with overlapping chunk padding,
  but a naive time-window filter ended up dropping some boundary segments entirely —
  worse than the original duplication issue. A text-similarity dedup approach would
  fix this properly but added more complexity than justified for a prototype, so I
  reverted to simple non-overlapping chunks and noted the trade-off here.
- **Environment setup**: ffmpeg wasn't picked up until VS Code was fully relaunched
  from a terminal with the updated PATH — a kernel restart alone wasn't enough.

#### Recommendations
- Use prompts for known vocabulary, but verify output rather than assume it's better.
- For production, use overlap + text-based dedup (not timestamp-based) to avoid
  content loss at boundaries.
- Chunk on natural pauses/silence rather than fixed intervals where possible.

#### Extension: Speaker Diarization (bonus)
Plain Whisper has no speaker awareness — it only outputs continuous text. Since plain Whisper has no speaker awareness, I explored adding a diarization layer:

## Diarization

In [ ]:
# Verify Hugging Face token
from huggingface_hub import whoami
print(whoami(token=hf_token))

{'type': 'user', 'id': '6a50b1716d7dc762c05e2bab', 'name': 'lou-PL-dev-HF', 'fullname': 'Lou Pls', 'email': 'louiseplessis@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1785542400, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/07AhRX_H1zu-WfvrvtIlz.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Speaker Diarization', 'role': 'read', 'createdAt': '2026-07-14T08:25:54.166Z'}}}


In [ ]:
# Load the speaker diarization pipeline from Hugging Face
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=hf_token
)

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/xvec_transform.npz: downloading bytes:           |  0.00B            

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

In [ ]:
# Convert MP3 to WAV for speaker diarization
from pydub import AudioSegment

mp3_path = "audio/california_moon_landing.mp3"
wav_path = "audio/california_moon_landing.wav"

audio = AudioSegment.from_mp3(mp3_path)
audio.export(wav_path, format="wav")
print(f"Converted to: {wav_path}")

Converted to: audio/california_moon_landing.wav


In [39]:
print("Speaker segments detected:")
for turn, _, speaker in diarization.speaker_diarization.itertracks(yield_label=True):
    print(f"[{turn.start:.1f}s - {turn.end:.1f}s] {speaker}")

Speaker segments detected:
[0.0s - 14.4s] SPEAKER_00
[4.9s - 5.2s] SPEAKER_01
[6.7s - 7.0s] SPEAKER_01
[9.8s - 10.6s] SPEAKER_01
[12.2s - 12.8s] SPEAKER_01
[15.1s - 23.5s] SPEAKER_00
[18.4s - 19.1s] SPEAKER_01
[24.0s - 25.5s] SPEAKER_00
[26.4s - 27.0s] SPEAKER_00
[27.6s - 31.6s] SPEAKER_00
[30.1s - 30.7s] SPEAKER_01
[31.8s - 35.1s] SPEAKER_00
[32.0s - 32.3s] SPEAKER_01
[35.7s - 47.2s] SPEAKER_00
[38.5s - 39.8s] SPEAKER_01
[39.8s - 39.9s] SPEAKER_01
[47.6s - 48.5s] SPEAKER_00
[49.0s - 50.4s] SPEAKER_00
[50.6s - 52.1s] SPEAKER_00
[52.3s - 53.0s] SPEAKER_00
[52.3s - 52.9s] SPEAKER_01
[54.0s - 57.7s] SPEAKER_00
[58.0s - 72.2s] SPEAKER_00
[58.5s - 60.0s] SPEAKER_01
[62.1s - 62.5s] SPEAKER_01
[63.6s - 64.9s] SPEAKER_01
[73.7s - 74.4s] SPEAKER_00
[73.8s - 74.1s] SPEAKER_01
[76.2s - 81.5s] SPEAKER_00
[77.0s - 78.6s] SPEAKER_01
[82.3s - 86.0s] SPEAKER_00


Speaker segments detected:
[0.0s - 14.4s] SPEAKER_00
[4.9s - 5.2s] SPEAKER_01
[6.7s - 7.0s] SPEAKER_01
[9.8s - 10.6s] SPEAKER_01
[12.2s - 12.8s] SPEAKER_01
[15.1s - 23.5s] SPEAKER_00
[18.4s - 19.1s] SPEAKER_01
[24.0s - 25.5s] SPEAKER_00
[26.4s - 27.0s] SPEAKER_00
[27.6s - 31.6s] SPEAKER_00
[30.1s - 30.7s] SPEAKER_01
[31.8s - 35.1s] SPEAKER_00
[32.0s - 32.3s] SPEAKER_01
[35.7s - 47.2s] SPEAKER_00
[38.5s - 39.8s] SPEAKER_01
[39.8s - 39.9s] SPEAKER_01
[47.6s - 48.5s] SPEAKER_00
[49.0s - 50.4s] SPEAKER_00
[50.6s - 52.1s] SPEAKER_00
[52.3s - 53.0s] SPEAKER_00
[52.3s - 52.9s] SPEAKER_01
[54.0s - 57.7s] SPEAKER_00
[58.0s - 72.2s] SPEAKER_00
[58.5s - 60.0s] SPEAKER_01
...
[73.8s - 74.1s] SPEAKER_01
[76.2s - 81.5s] SPEAKER_00
[77.0s - 78.6s] SPEAKER_01
[82.3s - 86.0s] SPEAKER_00


In [40]:
def get_speaker_for_segment(seg_start, seg_end, diarization_result):
    speaker_durations = {}
    for turn, _, speaker in diarization_result.itertracks(yield_label=True):
        overlap_start = max(seg_start, turn.start)
        overlap_end = min(seg_end, turn.end)
        overlap = max(0, overlap_end - overlap_start)
        if overlap > 0:
            speaker_durations[speaker] = speaker_durations.get(speaker, 0) + overlap
    if not speaker_durations:
        return "UNKNOWN"
    return max(speaker_durations, key=speaker_durations.get)

labeled_transcript = []
for seg in all_segments:
    speaker = get_speaker_for_segment(seg["start"], seg["end"], diarization.speaker_diarization)
    labeled_transcript.append({
        "start": seg["start"],
        "end": seg["end"],
        "speaker": speaker,
        "text": seg["text"].strip()
    })

print("Speaker-labeled transcript:\n")
for entry in labeled_transcript:
    print(f"[{entry['start']:.1f}s] {entry['speaker']}: {entry['text']}")

Speaker-labeled transcript:

[0.0s] SPEAKER_00: It was rather interesting just to watch them gathering their materials and bouncing around
[7.0s] SPEAKER_00: their, what do they call it, kangaroo walk or something like that.
[12.5s] SPEAKER_00: Who named it that?
[13.5s] SPEAKER_00: I don't know.
[14.5s] SPEAKER_00: I bet those men are going to get quite a reception when they get back to New York.
[19.0s] SPEAKER_00: Oh yes, I'll be so glad when they land back now.
[22.0s] SPEAKER_00: I think that's pretty well a fact because they've landed so many safely now.
[30.0s] SPEAKER_00: and I feel relief. Just getting off of the moon was the thing that was...
[35.0s] SPEAKER_00: Have they met with the one that was circling?
[38.0s] SPEAKER_00: Yes, they've rendezvoused. So I understand. That wasn't shown either, so I...
[44.0s] SPEAKER_00: But they say they have rendezvoused, so...
[47.0s] SPEAKER_00: That's a matter of making the circles and then coming down.
[53.0s] SPEAKER_00: What do you 

Speaker-labeled transcript:

[0.0s] SPEAKER_00: It was rather interesting just to watch them gathering their materials and bouncing around
[7.0s] SPEAKER_00: their, what do they call it, kangaroo walk or something like that.
[12.5s] SPEAKER_00: Who named it that?
[13.5s] SPEAKER_00: I don't know.
[14.5s] SPEAKER_00: I bet those men are going to get quite a reception when they get back to New York.
[19.0s] SPEAKER_00: Oh yes, I'll be so glad when they land back now.
[22.0s] SPEAKER_00: I think that's pretty well a fact because they've landed so many safely now.
[30.0s] SPEAKER_00: and I feel relief. Just getting off of the moon was the thing that was...
[35.0s] SPEAKER_00: Have they met with the one that was circling?
[38.0s] SPEAKER_00: Yes, they've rendezvoused. So I understand. That wasn't shown either, so I...
[44.0s] SPEAKER_00: But they say they have rendezvoused, so...
[47.0s] SPEAKER_00: That's a matter of making the circles and then coming down.
[53.0s] SPEAKER_00: What do you sort of imagine for the future? Do you imagine them sending a...
[57.0s] SPEAKER_00: I think they will. I think they will.
[60.0s] SPEAKER_00: And we'll do some more exploring up there.
[62.0s] SPEAKER_00: Very positive.
[64.0s] SPEAKER_00: Because that was such a very small area, when you think of it, that they just gathered rocks and, oh, samples of soil and all.
[78.0s] SPEAKER_00: And they did a probe for some samples.
[82.0s] SPEAKER_00: And just what was going to come of that, I don't know. I'd be glad to read it.

Conclusion:

The majority-overlap alignment strategy assigns each Whisper segment to whichever speaker talked for the most cumulative time within that segment's window. Because Whisper's segments are several seconds long while Speaker 1's turns are brief backchannel interjections nested inside Speaker 0's longer turns, Speaker 0 dominates every segment's overlap and the naive alignment collapses to a single speaker throughout. A more accurate approach would align at the word level (using Whisper's word-level timestamps) rather than the segment level, so a single short interjection isn't averaged away inside a multi-second segment.